In [24]:
import duckdb
import polars as pl
import pandas as pd
import json
import os
import numpy as np

In [58]:
from minecraft_data_dashboard.utils.db import (
    get_modrinth_overview_kpis,
    get_project_type_distribution,
    get_modrinth_project_listings_time_series,
    get_modrinth_overview_preview,
    get_mod_version_distribution,
    get_first_activity
)

In [25]:
modrinth_silver_path = "/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/silver/dev/api_modrinth_com.duckdb"
modrinth_bronze_path = "/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/api_modrinth_com.duckdb"
modrinth_gold_path = "/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/gold/dev/api_modrinth_com.duckdb"
mojang_version_manifest_db_path = "/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/silver/dev/piston_meta_mojang_com.duckdb"

In [56]:
with duckdb.connect(modrinth_silver_path) as silver_con:
    df_check = silver_con.execute('SELECT * FROM modrinth_project_versions LIMIT 100').fetch_df()
    
df_check
    

,run_id,project_type,project_id,version_id,hashed_payload,c_pull_timestamp_utc,author_id,name,version_number,changelog,...,dependencies,game_versions,version_type,loaders,featured,status,requested_status,date_published,downloads,files
0,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,1BZ68dI8,ol9uAtMf,fbc093a274364286d0caa3c34158aeff3f1a32c74d8588...,2026-08-26 05:51:48.540420-07:00,6NlRvOsq,Pegasus Vx (v0.3.3),0.3.3,None,...,[],"[1.19.2, 1.19.3, 1.19.4, 1.20, 1.20.1, 1.20.2,...",alpha,[iris],False,listed,None,2025-08-14 11:07:59.542189-07:00,681,"[{""id"":""vGqgcdXD"",""hashes"":{""sha512"":""b6cb7ebb..."
1,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,1BZ68dI8,j6kmC96s,e167afeffd2be47bc0e3f651618f634523cf2f74264ac8...,2026-08-26 05:51:48.540420-07:00,6NlRvOsq,Pegasus (0.3.0b),0.3.0b,None,...,[],"[1.19.2, 1.19.3, 1.19.4, 1.20, 1.20.1, 1.20.2,...",alpha,[iris],False,listed,None,2025-04-12 17:21:36.471812-07:00,2144,"[{""id"":""AnaA1hYA"",""hashes"":{""sha512"":""9094894e..."
2,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,1BZ68dI8,CrWpZCHu,4b6d21c3fb62019e93d9ad133492bd737bec394c4e43ba...,2026-08-26 05:51:48.540420-07:00,6NlRvOsq,Pegasus v_0.2.1,0.2.1,None,...,[],"[1.19.2, 1.19.3, 1.19.4, 1.20, 1.20.1, 1.20.2,...",alpha,[iris],False,listed,None,2025-02-05 17:28:30.205182-08:00,23608,"[{""id"":""HEFrT6N9"",""hashes"":{""sha512"":""0d10a97c..."
3,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,1BZ68dI8,DvUxEAb6,fe9554d65946c7dd12f9ec9f1ea2e71caf2ca0973bcb60...,2026-08-26 05:51:48.540420-07:00,6NlRvOsq,Pegasus v0.2.0,0.2.0,None,...,[],"[1.19.1, 1.19.2, 1.19.3, 1.19.4, 1.20, 1.20.1,...",alpha,[iris],False,listed,None,2025-02-05 13:43:34.074940-08:00,1562,"[{""id"":""foxF1XT7"",""hashes"":{""sha1"":""d38d5f651b..."
4,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,1BZ68dI8,7rY5E8Jp,875568f19f669fdfd97369bd103768c603f921447d582a...,2026-08-26 05:51:48.540420-07:00,6NlRvOsq,Pegasus Shaders v 0.1.2,0.1.2,None,...,[],"[1.19.2, 1.19.3, 1.19.4, 1.20, 1.20.1, 1.20.2,...",beta,[iris],False,listed,None,2024-11-19 13:14:40.320452-08:00,18363,"[{""id"":""xYzQbjBt"",""hashes"":{""sha1"":""a6a156f658..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,A1wQqzrz,EzC8VCUP,0ce40eafc12c5e9eef240dd37f32b4b3d51250e05dc54b...,2026-08-26 05:52:53.441202-07:00,pXNNL5g4,The only version,1,None,...,[],[1.19.4],release,[vanilla],False,listed,None,2023-04-24 05:04:04.463518-07:00,1819,"[{""id"":""w4XfBj7b"",""hashes"":{""sha512"":""61e4e564..."
96,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,ARW1Jej5,Ru1EPCkc,000fa96c0e91e3d9350533893e46d6177d1ebacdae9497...,2026-08-26 05:52:52.082667-07:00,7U2MCbSd,Major fixes and improvements,1.1-beta,None,...,[],"[1.21.1, 1.21.2, 1.21.3, 1.21.4, 1.21.5, 1.21....",beta,"[iris, optifine]",False,listed,None,2026-06-12 13:44:57.212719-07:00,220,"[{""id"":""LUEAqUlk"",""hashes"":{""sha512"":""a5ef0d34..."
97,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,ARW1Jej5,YuuwWKMW,1a422f644860decb371a09de478e9a3417808297d8532d...,2026-08-26 05:52:52.082667-07:00,7U2MCbSd,UNSTABLE,1.1-alpha,None,...,[],"[1.21.1, 1.21.2, 1.21.3, 1.21.4, 1.21.5, 1.21....",alpha,"[iris, optifine]",False,listed,None,2026-06-11 16:15:14.443412-07:00,50,"[{""id"":""mElOnZYY"",""hashes"":{""sha1"":""75eb1bd356..."
98,63a1d9ac-4724-44a2-98e8-7eb7bce86f23,shader,ARW1Jej5,ghbDY2Wq,765621e17ceb517f75d25937a1c7d260beaae6c55901fd...,2026-08-26 05:52:52.082667-07:00,7U2MCbSd,First release of Polace,1.0-beta,None,...,[],"[1.21.1, 1.21.2, 1.21.3, 1.21.4, 1.21.5, 1.21....",beta,"[iris, optifine]",False,listed,None,2026-06-10 14:31:53.051851-07:00,51,"[{""id"":""rYU525xY"",""hashes"":{""sha1"":""9751fa804c..."


In [60]:
with duckdb.connect(modrinth_gold_path, read_only=True) as conn:
    tables = conn.execute(
        """
        SHOW TABLES
        """
    ).fetchall()

    for table_row in tables:
        table_name = table_row[0]

        print("=" * 100)
        print(f"TABLE: {table_name}")
        print("=" * 100)

        schema_df = conn.execute(
            f"""
            DESCRIBE {table_name}
            """
        ).df()

        print(schema_df.to_string(index=False))
        print()

TABLE: modrinth_project_signals
              column_name              column_type null  key default extra
               project_id                  VARCHAR  YES None    None  None
             project_type                  VARCHAR  YES None    None  None
                    title                  VARCHAR  YES None    None  None
           current_run_id                  VARCHAR  YES None    None  None
        comparison_run_id                  VARCHAR  YES None    None  None
          baseline_run_id                  VARCHAR  YES None    None  None
      current_observed_at TIMESTAMP WITH TIME ZONE  YES None    None  None
   comparison_observed_at TIMESTAMP WITH TIME ZONE  YES None    None  None
     baseline_observed_at TIMESTAMP WITH TIME ZONE  YES None    None  None
        current_downloads                   BIGINT  YES None    None  None
     comparison_downloads                   BIGINT  YES None    None  None
       baseline_downloads                   BIGINT  YES None    None

In [ ]:
with duckdb.connect(
    modrinth_gold_path,
    read_only=True,
) as conn:

    activity_df = conn.execute(
        f"""
       SELECT
    category_momentum_rank,
    gameplay_category,

    total_projects,
    momentum_project_count,

    ROUND(
        momentum_project_share * 100,
        2
    ) AS momentum_project_share_percent,

    projects_with_strong_discovery,
    projects_with_strong_activity,
    projects_with_breakout_signal,
    projects_with_strong_audience,

    total_downloads_gained,
    momentum_downloads_gained,

    ROUND(
        momentum_download_gain_share * 100,
        2
    ) AS momentum_download_gain_share_percent,

    ROUND(
        average_momentum_downloads_gained,
        2
    ) AS average_momentum_downloads_gained,

    ROUND(
        median_velocity_ratio,
        2
    ) AS median_velocity_ratio,

    ROUND(
        momentum_count_percentile * 100,
        2
    ) AS momentum_count_percentile,

    ROUND(
        momentum_share_percentile * 100,
        2
    ) AS momentum_share_percentile,

    ROUND(
        momentum_download_gain_share_percentile * 100,
        2
    ) AS momentum_demand_percentile,

    ROUND(
        category_momentum_score,
        2
    ) AS category_momentum_score

FROM modrinth_category_momentum_signals

WHERE project_type = 'mod'
    AND is_category_momentum_candidate = TRUE

ORDER BY
    category_momentum_rank

LIMIT 30;
        """
    ).df()

activity_df

,gameplay_category,category_momentum_rank,total_projects,momentum_project_count,title,current_downloads,current_downloads_gained,discovery_score,development_activity_score,content_opportunity_score,content_opportunity_rank
0,utility,1,35818,118,Just Enough Items (JEI),74544467,527732,NaN,99.470587,99.470587,1
1,utility,1,35818,118,Simple Voice Chat,66715071,511970,82.350276,96.255131,96.860868,2
2,utility,1,35818,118,Project: MCSR Ranked,1992041,40226,95.409014,89.344119,96.312155,5
3,utility,1,35818,118,Tensura EP Scaling,26558,7556,96.068735,NaN,96.068735,8
4,utility,1,35818,118,Quests & Teams Fixes,21744,7851,95.965438,48.918593,95.965438,10
...,...,...,...,...,...,...,...,...,...,...,...
215,worldgen,10,8265,14,The Broken Script,2862098,55199,NaN,80.017224,81.928013,175
216,worldgen,10,8265,14,MVS - Moog's Voyager Structures,7802533,77433,81.854378,NaN,81.854378,177
217,worldgen,10,8265,14,YUNG's API,33534348,165572,NaN,81.300131,81.300131,188
218,worldgen,10,8265,14,Wizards (RPG Series),5206890,22535,9.811565,80.886093,80.886093,196


In [16]:
with duckdb.connect(modrinth_gold_path, read_only=True) as conn:
    activity_df = conn.execute("""
                               SELECT
    breakout_rank,
    project_id,
    title,

    current_downloads,
    comparison_downloads,
    current_downloads_gained,
    recent_growth_share,

    current_velocity,
    baseline_velocity,
    velocity_ratio,

    trend_rank,
    trend_score,

    trend_strength_percentile,
    underexposure_percentile,

    breakout_score

FROM modrinth_project_breakout_signals

WHERE is_breakout_candidate = TRUE

ORDER BY
    breakout_rank

LIMIT 30;""").df()
activity_df
    

,breakout_rank,project_id,title,current_downloads,comparison_downloads,current_downloads_gained,recent_growth_share,current_velocity,baseline_velocity,velocity_ratio,trend_rank,trend_score,trend_strength_percentile,underexposure_percentile,breakout_score
0,1,PsY5btQe,Potion API,24798,16727,8071,0.482513,125.096226,25.712723,4.865149,34,83.165222,0.977444,0.959672,96.847624
1,2,UJOmjHfx,Tensura: Ancient Artifacts,21822,14000,7822,0.558714,121.236858,63.191018,1.918577,45,81.745146,0.969925,0.963090,96.649509
2,3,9LFj5OCy,Tensura EP Scaling,26558,19002,7556,0.397642,117.113999,61.136072,1.915628,54,81.123665,0.963773,0.957621,96.068735
3,4,SWZGEk5x,Quests & Teams Fixes,21744,13893,7851,0.565105,121.686342,79.400337,1.532567,66,80.184014,0.955571,0.963773,95.965438
4,5,NysbPTkh,Legend of the Lucky Blocks,39189,27121,12068,0.444969,187.047609,39.855585,4.693134,15,88.747673,0.990431,0.926863,95.759282
5,6,PacPygzz,Hypixel Lucky Blocks,43907,33009,10898,0.330152,168.913229,33.454745,5.049007,16,87.278042,0.989747,0.916610,95.177544
6,7,PriJM6Wm,OPAC-UI,20560,13224,7336,0.554749,113.704115,75.745182,1.501140,97,78.692049,0.934381,0.967874,95.083297
7,8,k2XSGtXz,HE: HariEntity,12037,7121,4916,0.690352,76.195397,29.333342,2.597570,131,74.196055,0.911141,0.987013,94.756090
8,9,b0C4QetW,Stone Lucky Block,12125,7519,4606,0.612581,71.390561,13.124023,5.439686,136,73.568564,0.907724,0.986329,94.539554
9,10,KvsvCJwP,Christmas Lucky Block,32986,26722,6264,0.234414,97.088683,35.239152,2.755137,100,78.485302,0.932331,0.947368,93.978947


In [15]:
with duckdb.connect(modrinth_gold_path, read_only=True) as conn:
    activity_df = conn.execute("""SELECT
    activity_rank,
    project_id,
    title,

    latest_version_number,
    latest_version_type,
    latest_release_at,
    days_since_latest_release,

    active_release_days_last_30_days,
    active_release_days_last_90_days,

    current_downloads_gained,
    current_velocity,

    release_recency_percentile,
    current_velocity_percentile,

    activity_score,
    trend_rank

FROM modrinth_project_activity_signals

WHERE is_activity_candidate = TRUE

ORDER BY
    activity_rank

LIMIT 30;""").df()
activity_df
    

,activity_rank,project_id,title,latest_version_number,latest_version_type,latest_release_at,days_since_latest_release,active_release_days_last_30_days,active_release_days_last_90_days,current_downloads_gained,current_velocity,release_recency_percentile,current_velocity_percentile,activity_score,trend_rank
0,1,u6dRKJwZ,Just Enough Items (JEI),30.26.0.186,beta,2026-08-26 11:40:51.326227-05:00,9.333333,15,54,527732,8179.566531,0.996473,0.992945,99.470587,<NA>
1,2,9eGKb6K1,Simple Voice Chat,fabric-2.6.22+26.3-snapshot-10,alpha,2026-08-25 12:36:08.884965-05:00,10.291667,4,20,511970,7935.263878,0.934744,0.992063,96.255131,41
2,3,c3onkd5k,AsyncParticles,20.1.4.0-alpha.2,alpha,2026-08-26 03:16:55.016274-05:00,9.666667,10,34,114138,1769.078556,0.976190,0.947972,96.187421,<NA>
3,4,P7dR8mSH,Fabric API,0.158.2+26.3,beta,2026-08-25 09:46:28.968740-05:00,10.416667,6,22,1921825,29787.269768,0.923280,1.000000,96.011004,282
4,5,5CTSrY8X,GPUBooster,1.3.2-1.21-1.21.1-neo,beta,2026-08-26 08:03:29.368206-05:00,9.458333,1,5,79122,1226.349100,0.986332,0.930335,95.751535,<NA>
5,6,VSNURh3q,Concurrent Chunk Management Engine (Fabric),0.4.2-alpha.0.65+26.3-snap-10,alpha,2026-08-25 10:54:15.052913-05:00,10.375000,7,40,274994,4262.261373,0.926808,0.982363,95.377722,93
6,7,rOUBggPv,SuperMartijn642's Core Lib,1.1.24a-neoforge-mc26.2,release,2026-08-25 18:46:55.444291-05:00,10.041667,7,11,99588,1543.561262,0.955467,0.938272,94.679142,<NA>
7,8,7MoE34WK,Gnetum,4.5.4+26.2-neoforge,release,2026-08-25 19:45:07.111887-05:00,10.000000,2,9,78988,1224.272171,0.960758,0.929453,94.484659,<NA>
8,9,Xy8aRQKS,Physics Mod,3.2.1,beta,2026-08-25 15:42:55.499271-05:00,10.166667,6,20,111751,1732.081321,0.944004,0.945326,94.466444,<NA>
9,10,WKwQAwke,LibJF,26.2.2+forge,beta,2026-08-25 08:14:14.636342-05:00,10.458333,2,4,159843,2477.481853,0.917549,0.966490,94.138372,<NA>


In [ ]:
with duckdb.connect(modrinth_gold_path, read_only=True) as conn:
    fan_signals_df = conn.execute("""
                                  SELECT
    fan_affinity_rank,
    project_id,
    title,
    current_downloads,
    follows,
    follows_per_1000_downloads,
    follow_rate_percentile,
    follower_count_percentile,
    fan_affinity_score

FROM modrinth_project_fan_signals

WHERE is_fan_signal_candidate = TRUE

ORDER BY fan_affinity_rank;""").df()

In [6]:
fan_signals_df

,fan_affinity_rank,project_id,title,current_downloads,follows,follows_per_1000_downloads,follow_rate_percentile,follower_count_percentile,fan_affinity_score
0,1,74oZhrEe,The Roads More Travelled [TRMT],174517,1214,6.956342,0.982817,0.959255,97.089318
1,2,kjZCvAn6,No Man's Land,148668,1052,7.076170,0.983743,0.952104,96.766510
2,3,MGHiBUyJ,Etherology,23173,725,31.286411,1.000000,0.929725,96.358304
3,4,LtZfKhel,Simple Clouds,189662,1110,5.852517,0.971396,0.954574,96.291140
4,5,RQDFHc4w,Creatures and Beasts,36311,706,19.443144,0.999383,0.927873,96.230129
5,6,82nEyXEm,"The First City ($50,000 MrBeast Mod)",214649,1175,5.474053,0.966766,0.957506,96.211374
6,7,aT3Teaxa,Middle-earth,80677,718,8.899686,0.991974,0.929005,95.945770
7,8,ClzhCK3K,Fathoms,30991,623,20.102610,0.999486,0.917584,95.678532
8,9,Jjc1149h,RAT's Mischief,128726,791,6.144835,0.974997,0.936619,95.542296
9,10,a93H3mKU,Inspecio,117857,755,6.406068,0.977570,0.933326,95.493591


In [62]:
with duckdb.connect(modrinth_gold_path, read_only=True) as conn:
    signals_df = conn.execute("""SELECT
    trend_rank,
    project_id,
    title,
    current_downloads,
    current_downloads_gained,
    current_velocity,
    baseline_velocity,
    velocity_lift,
    velocity_ratio,
    velocity_ratio_percentile,
    download_gain_percentile,
    trend_score

FROM modrinth_project_signals

WHERE is_trending_candidate = TRUE

ORDER BY trend_rank

LIMIT 30;""").df()
signals_df

,trend_rank,project_id,title,current_downloads,current_downloads_gained,current_velocity,baseline_velocity,velocity_lift,velocity_ratio,velocity_ratio_percentile,download_gain_percentile,trend_score
0,1,I9W1u5Ac,Project: MCSR Ranked,1992041,40226,623.481698,264.138224,359.343473,2.360437,0.975393,0.933698,95.409014
1,2,AibBIVmj,OneConfig,557197,34610,536.436672,193.481480,342.955192,2.772548,0.981545,0.917293,94.833195
2,3,y6DuFGwJ,Skyblocker • Hypixel Skyblock,4566123,63682,987.037276,633.280142,353.757134,1.558611,0.930964,0.959672,94.509988
3,4,jJJLywXp,Odin - Hypixel Skyblock,1427513,34786,539.164579,306.273244,232.891334,1.760404,0.962406,0.918660,94.002448
4,5,IMuO8COj,[ESF] Entity Sound Features,2567675,28437,440.758441,236.197871,204.560570,1.866056,0.967874,0.904990,93.537626
5,6,fB1gJLGX,e4all,153369,23316,361.385653,89.582967,271.802686,4.034089,0.990431,0.884484,93.446389
6,7,UR0ocuEt,NBT Autocomplete,1400184,28400,440.184960,250.001119,190.183841,1.760732,0.963090,0.904306,93.277267
7,8,AtOSAunf,Optimized Block Entities [OBE],1011781,67042,1039.115497,761.958882,277.156615,1.363742,0.889952,0.961039,92.413052
8,9,eHe8QhtD,Compose Multiplatform,144741,17616,273.038671,91.436447,181.602224,2.986103,0.983595,0.858510,91.680576
9,10,2Uev7LdA,LambdaBetterGrass,1650706,18272,283.206324,166.830504,116.375820,1.697569,0.958988,0.863295,90.862890


In [8]:
import duckdb

modrinth_silver_path = (
    "/Users/admin/AroTekCodingSpace/Python-Workspace/"
    "Minecraft-Data-Platform/data/silver/dev/api_modrinth_com.duckdb"
)

with duckdb.connect(modrinth_silver_path, read_only=True) as conn:
    tables = conn.execute(
        """
        SHOW TABLES
        """
    ).fetchall()

    for table_row in tables:
        table_name = table_row[0]

        print("=" * 100)
        print(f"TABLE: {table_name}")
        print("=" * 100)

        schema_df = conn.execute(
            f"""
            DESCRIBE {table_name}
            """
        ).df()

        print(schema_df.to_string(index=False))
        print()

TABLE: modrinth_categories
         column_name              column_type null  key default extra
              header                  VARCHAR  YES None    None  None
                icon                  VARCHAR  YES None    None  None
                name                  VARCHAR  YES None    None  None
        project_type                  VARCHAR  YES None    None  None
              run_id                  VARCHAR  YES None    None  None
      hashed_payload                  VARCHAR  YES None    None  None
c_pull_timestamp_utc TIMESTAMP WITH TIME ZONE  YES None    None  None

TABLE: modrinth_game_versions
         column_name              column_type null  key default extra
                date                  VARCHAR  YES None    None  None
               major                  BOOLEAN  YES None    None  None
             version                  VARCHAR  YES None    None  None
        version_type                  VARCHAR  YES None    None  None
              run_id            

### Trend detection

In [26]:
project_download_metrics_query = """
    WITH run_summary AS (
        SELECT
            run_id,
            MAX(c_pull_timestamp_utc) AS observed_at

        FROM modrinth_project_listings

        WHERE project_type = 'mod'

        GROUP BY
            run_id
    ),

    ranked_runs AS (
        SELECT
            run_id,
            observed_at,

            ROW_NUMBER() OVER (
                ORDER BY
                    observed_at DESC,
                    run_id DESC
            ) AS run_rank

        FROM run_summary
    ),

    selected_runs AS (
        SELECT
            run_id,
            observed_at,
            run_rank

        FROM ranked_runs

        WHERE run_rank <= 30
    ),

    project_snapshots AS (
        SELECT
            p.run_id,
            r.run_rank,
            r.observed_at,

            p.project_id,

            JSON_EXTRACT_STRING(
                p.payload,
                '$.title'
            ) AS title,

            TRY_CAST(
                JSON_EXTRACT_STRING(
                    p.payload,
                    '$.downloads'
                )
                AS BIGINT
            ) AS total_downloads,

            p.c_pull_timestamp_utc,

            ROW_NUMBER() OVER (
                PARTITION BY
                    p.run_id,
                    p.project_id

                ORDER BY
                    p.c_pull_timestamp_utc DESC
            ) AS row_num

        FROM modrinth_project_listings AS p

        INNER JOIN selected_runs AS r
            ON p.run_id = r.run_id

        WHERE p.project_type = 'mod'
    ),

    deduped_snapshots AS (
        SELECT
            run_id,
            run_rank,
            observed_at,
            project_id,
            title,
            total_downloads

        FROM project_snapshots

        WHERE row_num = 1
    ),

    with_previous AS (
        SELECT
            *,

            LAG(run_id) OVER (
                PARTITION BY project_id
                ORDER BY observed_at
            ) AS previous_run_id,

            LAG(total_downloads) OVER (
                PARTITION BY project_id
                ORDER BY observed_at
            ) AS previous_total_downloads,

            LAG(observed_at) OVER (
                PARTITION BY project_id
                ORDER BY observed_at
            ) AS previous_observed_at

        FROM deduped_snapshots
    ),

    interval_metrics AS (
        SELECT
            *,

            total_downloads
                - previous_total_downloads
                AS downloads_gained,

            DATE_DIFF(
                'second',
                previous_observed_at,
                observed_at
            ) / 3600.0
                AS hours_since_previous_snapshot

        FROM with_previous
    )

    SELECT
        run_id,
        run_rank,
        observed_at,

        project_id,
        title,

        total_downloads,
        previous_total_downloads,
        downloads_gained,

        hours_since_previous_snapshot,

        downloads_gained
            / NULLIF(
                hours_since_previous_snapshot,
                0
            ) AS download_velocity

    FROM interval_metrics

    WHERE previous_total_downloads IS NOT NULL

    ORDER BY
        observed_at DESC,
        download_velocity DESC
"""

with duckdb.connect(modrinth_bronze_path, read_only=True) as bronze_con:
    df = bronze_con.execute(project_download_metrics_query).fetch_df()
    
df.head(25)
    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,run_id,run_rank,observed_at,project_id,title,total_downloads,previous_total_downloads,downloads_gained,hours_since_previous_snapshot,download_velocity
0,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,P7dR8mSH,Fabric API,247786665,245864840,1921825,64.518333,29787.269768
1,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,AANobbMI,Sodium,220091956,218493006,1598950,64.518333,24782.878252
2,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,NNAgCjsB,Entity Culling,161859058,160655791,1203267,64.518333,18650.001292
3,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,YL57xq9U,Iris Shaders,171285854,170092881,1192973,64.518333,18490.449743
4,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,9s6osm5g,Cloth Config API,163416736,162414763,1001973,64.518333,15530.050890
5,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,OhduvhIc,VeinMiner,79381136,78434372,946764,64.518333,14674.340627
6,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,uXXizFIs,FerriteCore,147035646,146173009,862637,64.518333,13370.416677
7,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,mOgUt4GM,Mod Menu,140006599,139176104,830495,64.518333,12872.232699
8,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,dxa0Bm8m,VeinMiner Hotkey,54269255,53438919,830336,64.518333,12869.768283
9,7fd9fb62-16c2-41c9-99e0-f87dae832968,1,2026-09-04 17:50:20.368359-07:00,Ha28R6CL,Fabric Language Kotlin,114545146,113735351,809795,64.518333,12551.393661


In [27]:
project_download_metrics_df = (
    df
    .sort_values(
        [
            "project_id",
            "observed_at",
        ]
    )
    .copy()
)

In [28]:
project_download_metrics_df["previous_velocity"] = (
    project_download_metrics_df
    .groupby("project_id")["download_velocity"]
    .shift(1)
)

In [29]:
project_download_metrics_df["baseline_velocity"] = (
    project_download_metrics_df
    .groupby("project_id")["download_velocity"]
    .transform(
        lambda series: (
            series
            .shift(1)
            .rolling(
                window=7,
                min_periods=3,
            )
            .median()
        )
    )
)

In [30]:
project_download_metrics_df["velocity_ratio"] = np.where(
    project_download_metrics_df["baseline_velocity"] > 0,
    (
        project_download_metrics_df["download_velocity"]
        / project_download_metrics_df["baseline_velocity"]
    ),
    np.nan,
)

project_download_metrics_df["acceleration"] = np.where(
    project_download_metrics_df["previous_velocity"] > 0,
    (
        project_download_metrics_df["download_velocity"]
        / project_download_metrics_df["previous_velocity"]
    ),
    np.nan,
)

In [31]:
latest_observed_at = (
    project_download_metrics_df["observed_at"].max()
)

latest_trend_metrics_df = (
    project_download_metrics_df[
        project_download_metrics_df["observed_at"]
        == latest_observed_at
    ]
    .copy()
)

In [32]:
latest_trend_metrics_df = (
    latest_trend_metrics_df[
        latest_trend_metrics_df["baseline_velocity"].notna()
    ]
)

In [33]:
trending_mods_df = (
    latest_trend_metrics_df
    .sort_values(
        "velocity_ratio",
        ascending=False,
    )
)

In [34]:
trending_mods_df[
    [
        "project_id",
        "title",
        "total_downloads",
        "downloads_gained",
        "download_velocity",
        "previous_velocity",
        "baseline_velocity",
        "velocity_ratio",
        "acceleration",
    ]
].head(30)

,project_id,title,total_downloads,downloads_gained,download_velocity,previous_velocity,baseline_velocity,velocity_ratio,acceleration
3416,SIG65WOL,MaceBot,80375,1877,29.092506,0.0,0.819579,35.496898,NaN
22115,8L2Wendd,afkmon,76,38,0.588980,0.0,0.049047,12.008543,NaN
19188,2grfsR75,AdvancedPlayTime [1.21 - 1.26],3286,50,0.774974,0.0,0.070236,11.033901,NaN
19341,681HeTaF,Colored Bricks Mod,4944,49,0.759474,0.0,0.070236,10.813223,NaN
17356,8XUM9Lwf,Flint Required,3651,62,0.960967,0.0,0.095980,10.012210,NaN
19433,Vw4gxpH7,Spell Powered Weapons,128,49,0.759474,0.0,0.095509,7.951862,NaN
9986,gE1dmuDj,Sculk And Jaw,2325,200,3.099894,0.0,0.458453,6.761644,NaN
9065,gXy6YNys,Nourished,2960,245,3.797370,0.0,0.575877,6.594063,NaN
9342,F5ao3BYo,Crystals Overhauled,35622,230,3.564878,0.0,0.584226,6.101883,NaN
34209,fBT7OYMU,NPC Mod,36,18,0.278990,0.0,0.045981,6.067474,NaN


In [35]:
breakout_mods_df = (
    latest_trend_metrics_df[
        (latest_trend_metrics_df["baseline_velocity"] <= 0)
        & (latest_trend_metrics_df["downloads_gained"] >= 1_000)
    ]
    .sort_values(
        "downloads_gained",
        ascending=False,
    )
)

In [36]:
breakout_mods_df[
    [
        "project_id",
        "title",
        "total_downloads",
        "downloads_gained",
        "download_velocity",
        "previous_velocity",
        "baseline_velocity",
    ]
].head(30)

,project_id,title,total_downloads,downloads_gained,download_velocity,previous_velocity,baseline_velocity
1,AANobbMI,Sodium,220091956,1598950,24782.878252,0.0,0.0
2,NNAgCjsB,Entity Culling,161859058,1203267,18650.001292,0.0,0.0
3,YL57xq9U,Iris Shaders,171285854,1192973,18490.449743,0.0,0.0
4,9s6osm5g,Cloth Config API,163416736,1001973,15530.050890,0.0,0.0
5,OhduvhIc,VeinMiner,79381136,946764,14674.340627,0.0,0.0
6,uXXizFIs,FerriteCore,147035646,862637,13370.416677,0.0,0.0
7,mOgUt4GM,Mod Menu,140006599,830495,12872.232699,0.0,0.0
8,dxa0Bm8m,VeinMiner Hotkey,54269255,830336,12869.768283,0.0,0.0
9,Ha28R6CL,Fabric Language Kotlin,114545146,809795,12551.393661,0.0,0.0
10,1eAoo2KR,YetAnotherConfigLib (YACL),119039711,761108,11796.770944,0.0,0.0


In [37]:
projects_to_inspect = [
    "AANobbMI",  # Sodium
    "SIG65WOL",  # MaceBot
    "I9W1u5Ac",  # Project: MCSR Ranked
    "Xy8aRQKS",  # Physics Mod
]

project_history_df = (
    project_download_metrics_df[
        project_download_metrics_df["project_id"].isin(
            projects_to_inspect
        )
    ]
    [
        [
            "project_id",
            "title",
            "observed_at",
            "total_downloads",
            "previous_total_downloads",
            "downloads_gained",
            "hours_since_previous_snapshot",
            "download_velocity",
            "previous_velocity",
            "baseline_velocity",
            "velocity_ratio",
        ]
    ]
    .sort_values(
        [
            "project_id",
            "observed_at",
        ]
    )
)

project_history_df

,project_id,title,observed_at,total_downloads,previous_total_downloads,downloads_gained,hours_since_previous_snapshot,download_velocity,previous_velocity,baseline_velocity,velocity_ratio
812223,AANobbMI,Sodium,2026-08-21 18:49:04.682461-07:00,211656843,211008015,648828,1.749722,370817.717098,NaN,NaN,NaN
742650,AANobbMI,Sodium,2026-08-21 18:56:59.493730-07:00,211656843,211656843,0,0.131944,0.000000,370817.717098,NaN,NaN
669852,AANobbMI,Sodium,2026-08-21 19:07:50.310225-07:00,211656843,211656843,0,0.180833,0.000000,0.000000,NaN,NaN
594168,AANobbMI,Sodium,2026-08-24 17:25:04.729335-07:00,213698656,211656843,2041813,70.287222,29049.561719,0.000000,0.000000,NaN
524352,AANobbMI,Sodium,2026-08-24 21:46:49.053741-07:00,213698656,213698656,0,4.362500,0.000000,29049.561719,14524.780859,0.000000
451132,AANobbMI,Sodium,2026-08-25 00:13:14.438090-07:00,213698656,213698656,0,2.440278,0.000000,0.000000,0.000000,NaN
378331,AANobbMI,Sodium,2026-08-25 14:27:30.458835-07:00,213698656,213698656,0,14.237778,0.000000,0.000000,0.000000,NaN
301496,AANobbMI,Sodium,2026-08-25 19:35:36.229194-07:00,214315937,213698656,617281,5.135000,120210.516066,0.000000,0.000000,NaN
231723,AANobbMI,Sodium,2026-08-26 06:00:44.010241-07:00,214315937,214315937,0,10.418889,0.000000,120210.516066,0.000000,NaN
221255,AANobbMI,Sodium,2026-08-27 21:11:46.071527-07:00,215469572,214315937,1153635,39.183889,29441.564702,0.000000,0.000000,NaN


In [38]:
run_interval_df = (
    project_download_metrics_df[
        [
            "run_id",
            "run_rank",
            "observed_at",
        ]
    ]
    .drop_duplicates()
    .sort_values("observed_at")
)

run_interval_df["hours_since_previous_run"] = (
    run_interval_df["observed_at"]
    .diff()
    .dt.total_seconds()
    / 3600
)

run_interval_df

,run_id,run_rank,observed_at,hours_since_previous_run
831182,f7d7af7d-e9a1-4bfe-9649-86faf2dcdcb4,13,2026-08-21 18:49:04.682461-07:00,NaN
746387,51dc50b7-6533-4816-bb25-2cc17c3d9dc9,12,2026-08-21 18:56:59.493730-07:00,0.131892
673242,21f1554d-0f8b-423c-b708-97f80453e17f,11,2026-08-21 19:07:50.310225-07:00,0.180782
609767,3b497402-8609-42e3-9304-38aadd1066f4,10,2026-08-24 17:25:04.729335-07:00,70.287339
530244,807ef283-b059-48b3-b90c-3f3f1dde2396,9,2026-08-24 21:46:49.053741-07:00,4.362312
457075,0a0fa12b-0c4d-415c-bad1-9d83d97629e4,8,2026-08-25 00:13:14.438090-07:00,2.440385
384481,8af5da09-b203-4d7e-bcab-e78002bb8f35,7,2026-08-25 14:27:30.458835-07:00,14.237784
320193,714895d5-1e06-4159-861f-77973a3476b8,6,2026-08-25 19:35:36.229194-07:00,5.134936
236916,8235114d-455f-4360-bb4d-dc5c73bc33ff,5,2026-08-26 06:00:44.010241-07:00,10.418828
223875,e487bcd0-ffbf-4078-b381-e796fdd81418,4,2026-08-27 21:11:46.071527-07:00,39.183906


In [39]:
trend_metrics_query = """
    WITH run_summary AS (
        SELECT
            run_id,
            MAX(c_pull_timestamp_utc) AS observed_at,
            COUNT(DISTINCT project_id) AS project_count

        FROM modrinth_project_listings

        WHERE project_type = 'mod'

        GROUP BY
            run_id
    ),

    current_run AS (
        SELECT
            run_id,
            observed_at

        FROM run_summary

        ORDER BY
            observed_at DESC,
            run_id DESC

        LIMIT 1
    ),

    comparison_run AS (
        SELECT
            r.run_id,
            r.observed_at

        FROM run_summary AS r

        CROSS JOIN current_run AS c

        WHERE r.observed_at <= (
            c.observed_at - INTERVAL '24 hours'
        )

        ORDER BY
            r.observed_at DESC,
            r.run_id DESC

        LIMIT 1
    ),

    baseline_run AS (
        SELECT
            r.run_id,
            r.observed_at

        FROM run_summary AS r

        CROSS JOIN comparison_run AS c

        WHERE r.observed_at <= (
            c.observed_at - INTERVAL '7 days'
        )

        ORDER BY
            r.observed_at DESC,
            r.run_id DESC

        LIMIT 1
    ),

    selected_runs AS (
        SELECT
            run_id,
            observed_at,
            'current' AS snapshot

        FROM current_run

        UNION ALL

        SELECT
            run_id,
            observed_at,
            'comparison' AS snapshot

        FROM comparison_run

        UNION ALL

        SELECT
            run_id,
            observed_at,
            'baseline' AS snapshot

        FROM baseline_run
    ),

    project_snapshots AS (
        SELECT
            s.snapshot,
            s.observed_at,

            p.project_id,

            JSON_EXTRACT_STRING(
                p.payload,
                '$.title'
            ) AS title,

            TRY_CAST(
                JSON_EXTRACT_STRING(
                    p.payload,
                    '$.downloads'
                )
                AS BIGINT
            ) AS total_downloads,

            ROW_NUMBER() OVER (
                PARTITION BY
                    s.snapshot,
                    p.project_id

                ORDER BY
                    p.c_pull_timestamp_utc DESC
            ) AS row_num

        FROM modrinth_project_listings AS p

        INNER JOIN selected_runs AS s
            ON p.run_id = s.run_id

        WHERE p.project_type = 'mod'
    ),

    deduped_snapshots AS (
        SELECT
            snapshot,
            observed_at,
            project_id,
            title,
            total_downloads

        FROM project_snapshots

        WHERE row_num = 1
    ),

    pivoted_snapshots AS (
        SELECT
            project_id,

            MAX(title) FILTER (
                WHERE snapshot = 'current'
            ) AS title,

            MAX(total_downloads) FILTER (
                WHERE snapshot = 'current'
            ) AS current_downloads,

            MAX(total_downloads) FILTER (
                WHERE snapshot = 'comparison'
            ) AS comparison_downloads,

            MAX(total_downloads) FILTER (
                WHERE snapshot = 'baseline'
            ) AS baseline_downloads,

            MAX(observed_at) FILTER (
                WHERE snapshot = 'current'
            ) AS current_observed_at,

            MAX(observed_at) FILTER (
                WHERE snapshot = 'comparison'
            ) AS comparison_observed_at,

            MAX(observed_at) FILTER (
                WHERE snapshot = 'baseline'
            ) AS baseline_observed_at

        FROM deduped_snapshots

        GROUP BY
            project_id
    ),

    calculated_metrics AS (
        SELECT
            *,

            current_downloads
                - comparison_downloads
                AS current_downloads_gained,

            comparison_downloads
                - baseline_downloads
                AS baseline_downloads_gained,

            DATE_DIFF(
                'second',
                comparison_observed_at,
                current_observed_at
            ) / 3600.0
                AS current_hours,

            DATE_DIFF(
                'second',
                baseline_observed_at,
                comparison_observed_at
            ) / 3600.0
                AS baseline_hours

        FROM pivoted_snapshots

        WHERE current_downloads IS NOT NULL
          AND comparison_downloads IS NOT NULL
          AND baseline_downloads IS NOT NULL
    )

    SELECT
        project_id,
        title,

        current_downloads,

        current_downloads_gained,
        baseline_downloads_gained,

        current_hours,
        baseline_hours,

        current_downloads_gained
            / NULLIF(current_hours, 0)
            AS current_velocity,

        baseline_downloads_gained
            / NULLIF(baseline_hours, 0)
            AS baseline_velocity,

        (
            current_downloads_gained
            / NULLIF(current_hours, 0)
        )
        /
        NULLIF(
            baseline_downloads_gained
            / NULLIF(baseline_hours, 0),
            0
        ) AS velocity_ratio

    FROM calculated_metrics

    WHERE current_downloads_gained >= 0
      AND baseline_downloads_gained >= 0

    ORDER BY
        velocity_ratio DESC
"""

In [42]:
with duckdb.connect(modrinth_bronze_path, read_only=True) as bronze_con:
    trend_metrics_df = bronze_con.execute(trend_metrics_query).fetch_df()

In [43]:
trend_metrics_df.head(30)

,project_id,title,current_downloads,current_downloads_gained,baseline_downloads_gained,current_hours,baseline_hours,current_velocity,baseline_velocity,velocity_ratio
0,UyHm1t8d,Random Respawn,1250,854,10,64.518333,173.727222,13.236548,0.057562,229.954867
1,yXbLlSQN,Scaling Mobs,3117,875,23,64.518333,173.727222,13.562037,0.132391,102.438911
2,561gHy1d,TabAPI,5903,825,43,64.518333,173.727222,12.787063,0.247514,51.661883
3,L4MwdPaO,Attribute Backport,7122,929,50,64.518333,173.727222,14.399008,0.287808,50.029993
4,6dCeBfdr,ChatTweaker,260,16,1,64.518333,173.727222,0.247992,0.005756,43.082879
5,NtdHRxOb,Infrastructury,2972,61,4,64.518333,173.727222,0.945468,0.023025,41.063369
6,tgz8bnNz,Totem on Death,434,30,2,64.518333,173.727222,0.464984,0.011512,40.390199
7,VwyK8jq2,Arsonage,154,15,1,64.518333,173.727222,0.232492,0.005756,40.390199
8,bi7rhFz3,SkillZ,5119,867,60,64.518333,173.727222,13.438041,0.345369,38.909225
9,PkYtepFd,Funny Graveyards,79,14,1,64.518333,173.727222,0.216993,0.005756,37.697519


In [44]:
trending_mods_df = (
    trend_metrics_df[
        (trend_metrics_df["current_downloads_gained"] >= 1_000)
        & (trend_metrics_df["baseline_velocity"] > 0)
    ]
    .sort_values(
        "velocity_ratio",
        ascending=False,
    )
)

In [45]:
trending_mods_df[
    [
        "project_id",
        "title",
        "current_downloads",
        "current_downloads_gained",
        "current_velocity",
        "baseline_velocity",
        "velocity_ratio",
    ]
].head(30)

,project_id,title,current_downloads,current_downloads_gained,current_velocity,baseline_velocity,velocity_ratio
255,s4WQHerM,3D Maps,6191,3774,58.495001,5.382001,10.868635
969,j9jxyR07,Yahiraft,4601,2674,41.445584,5.658296,7.324747
1254,JWEYmR12,Pastek's Lucky Block,10494,4196,65.035778,10.746733,6.051679
1293,ROwCMI9N,Integrated Lucky Blocks,21559,4066,63.020847,10.821563,5.823637
1294,bxNeVD9o,Redecorate,8681,1103,17.095916,2.947149,5.800832
1323,b0C4QetW,Stone Lucky Block,12125,4606,71.390561,13.124023,5.439686
2443,TsQgSXVF,The Forest Anomaly,19235,1011,15.669965,2.987442,5.245278
2473,PacPygzz,Hypixel Lucky Blocks,43907,10898,168.913229,33.454745,5.049007
2482,ETYqWHa5,Dungeons+: More Spawners,83609,1032,15.995453,3.171639,5.043277
2526,PsY5btQe,Potion API,24798,8071,125.096226,25.712723,4.865149


In [46]:
trend_candidates_df = (
    trend_metrics_df[
        (trend_metrics_df["baseline_velocity"] > 0)
        & (trend_metrics_df["current_downloads_gained"] >= 1_000)
        & (trend_metrics_df["velocity_ratio"] > 1)
    ]
    .copy()
)

In [47]:
trend_candidates_df["velocity_ratio_percentile"] = (
    trend_candidates_df["velocity_ratio"]
    .rank(
        pct=True,
        method="average",
    )
)

trend_candidates_df["download_gain_percentile"] = (
    trend_candidates_df["current_downloads_gained"]
    .rank(
        pct=True,
        method="average",
    )
)

In [48]:
trend_candidates_df["trend_score"] = (
    100
    * (
        2
        * trend_candidates_df["velocity_ratio_percentile"]
        * trend_candidates_df["download_gain_percentile"]
    )
    / (
        trend_candidates_df["velocity_ratio_percentile"]
        + trend_candidates_df["download_gain_percentile"]
    )
)

In [49]:
trending_ranked_df = (
    trend_candidates_df
    .sort_values(
        "trend_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

trending_ranked_df["trend_rank"] = (
    trending_ranked_df.index + 1
)

In [50]:
trending_ranked_df["velocity_lift"] = (
    trending_ranked_df["current_velocity"]
    - trending_ranked_df["baseline_velocity"]
)

In [51]:
trending_ranked_df[
    [
        "trend_rank",
        "project_id",
        "title",
        "current_downloads",
        "current_downloads_gained",
        "current_velocity",
        "baseline_velocity",
        "velocity_ratio",
        "velocity_ratio_percentile",
        "velocity_lift",
        "download_gain_percentile",
        "trend_score",
    ]
].head(30)

,trend_rank,project_id,title,current_downloads,current_downloads_gained,current_velocity,baseline_velocity,velocity_ratio,velocity_ratio_percentile,velocity_lift,download_gain_percentile,trend_score
0,1,I9W1u5Ac,Project: MCSR Ranked,1992041,40226,623.481698,264.138224,2.360437,0.975393,359.343473,0.933698,95.409014
1,2,AibBIVmj,OneConfig,557197,34610,536.436672,193.481480,2.772548,0.981545,342.955192,0.917293,94.833195
2,3,y6DuFGwJ,Skyblocker • Hypixel Skyblock,4566123,63682,987.037276,633.280142,1.558611,0.930964,353.757134,0.959672,94.509988
3,4,jJJLywXp,Odin - Hypixel Skyblock,1427513,34786,539.164579,306.273244,1.760404,0.962406,232.891334,0.918660,94.002448
4,5,IMuO8COj,[ESF] Entity Sound Features,2567675,28437,440.758441,236.197871,1.866056,0.967874,204.560570,0.904990,93.537626
5,6,fB1gJLGX,e4all,153369,23316,361.385653,89.582967,4.034089,0.990431,271.802686,0.884484,93.446389
6,7,UR0ocuEt,NBT Autocomplete,1400184,28400,440.184960,250.001119,1.760732,0.963090,190.183841,0.904306,93.277267
7,8,AtOSAunf,Optimized Block Entities [OBE],1011781,67042,1039.115497,761.958882,1.363742,0.889952,277.156615,0.961039,92.413052
8,9,eHe8QhtD,Compose Multiplatform,144741,17616,273.038671,91.436447,2.986103,0.983595,181.602224,0.858510,91.680576
9,10,2Uev7LdA,LambdaBetterGrass,1650706,18272,283.206324,166.830504,1.697569,0.958988,116.375820,0.863295,90.862890
